# Dataset B — Random Forest baseline (RF-01)

Run five-fold cross-validation only on the existing Train partition of split_v1, using the experimental temporary training feature lock v0.2. Use the existing tree profile matrices from dataset_b_preprocessing_v2. Do not refit preprocessing, tune hyperparameters, or use Validation/Test.

The first code cell verifies the source contracts and all five folds before any model fit. The second cell fits one new RF per fold, then writes cv_results.json and cv_oof_predictions.csv under artifacts/dataset_b/rf_baseline_v1 only after all folds succeed.

Status: this restored notebook has not been executed. The interrupted earlier run did not save fold artifacts. If execution stops, restart CV from fold 1.


## 1. Validate the locked Train folds

Load the governing contracts and Train partition, rebuild the locked splitter, and verify every row position, record ID, label, matrix shape and finite value before any estimator is created.

In [2]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import sys
import time
import warnings

import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "config/dataset_b/model_protocol_v1.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate Dataset B project root.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if Path(sys.prefix).name.lower() != "credit-risk":
    raise RuntimeError("Select the credit-risk Python environment before running RF-01.")

from src.dataset_b.preprocessing import DatasetBSplitLoader

config_dir = PROJECT_ROOT / "config/dataset_b"
protocol_path = config_dir / "model_protocol_v1.yaml"
feature_path = config_dir / "feature_set_v1.yaml"
split_path = config_dir / "split_v1.yaml"
with protocol_path.open(encoding="utf-8") as f:
    protocol = yaml.safe_load(f)
with feature_path.open(encoding="utf-8") as f:
    feature_contract = yaml.safe_load(f)
with split_path.open(encoding="utf-8") as f:
    split_contract = yaml.safe_load(f)
cv_root = PROJECT_ROOT / protocol["contracts"]["cv_artifact"]
cv_manifest_path = cv_root / "manifest.json"
with cv_manifest_path.open(encoding="utf-8") as f:
    cv_manifest = json.load(f)

if protocol["dataset"] != "dataset_b" or protocol["protocol_version"] != "model_protocol_v1":
    raise ValueError("Unexpected model protocol.")
if protocol["contracts"]["preprocessing_version"] != "dataset_b_preprocessing_v2":
    raise ValueError("Unexpected preprocessing contract.")
if feature_contract["feature_version"] != cv_manifest["feature_version"] or feature_contract["feature_version"] != "temporary_training_lock_v0_2":
    raise ValueError("Feature version mismatch.")
if split_contract["split_version"] != cv_manifest["split_version"] or split_contract["split_version"] != "split_v1":
    raise ValueError("Split version mismatch.")
if cv_manifest["preprocessing_version"] != "dataset_b_preprocessing_v2":
    raise ValueError("Unexpected preprocessing version.")
if (cv_manifest["source_partition"], cv_manifest["fit_scope"], cv_manifest["use"]) != (
    "train", "fold_train_only", "cross_validation_only"
):
    raise ValueError("CV preprocessing must be fold-local on Train.")
if cv_manifest["encoded_feature_count"] != 66 or len(cv_manifest["output_columns"]) != 66:
    raise ValueError("Expected 66 encoded columns.")
if feature_contract["constraints"]["encoded_feature_count"] != 66:
    raise ValueError("Feature contract mismatch.")
if not set(cv_manifest["output_raw_sources"]) <= set(feature_contract["model_input_raw_columns"]):
    raise ValueError("CV matrices use a forbidden raw input.")
if "rf" not in cv_manifest["tree_profile_shared_by"]:
    raise ValueError("Tree profile is not declared for RF.")
expected_cv = {"splitter": "StratifiedKFold", "n_splits": 5, "shuffle": True, "random_state": 42}
for key, value in expected_cv.items():
    if protocol["cross_validation"][key] != value or split_contract["cross_validation"][key] != value:
        raise ValueError(f"CV contract mismatch: {key}.")
if (
    protocol["cross_validation"]["population"] != "train_only"
    or split_contract["cross_validation"]["allowed_population"] != "train_only"
    or protocol["cross_validation"]["hyperparameter_search"] is not False
    or protocol["cross_validation"]["trial_budget_per_model"] != 1
    or protocol["cross_validation"]["primary_metric"] != "roc_auc"
):
    raise ValueError("Unexpected CV scope or objective.")
rf_contract = protocol["models"]["rf"]
rf_parameters = dict(rf_contract["parameters"])
if (
    rf_contract["estimator"] != "sklearn.ensemble.RandomForestClassifier"
    or rf_contract["preprocessing_profile"] != "tree"
    or rf_parameters["class_weight"] is not None
    or set(rf_parameters) != {
        "n_estimators", "criterion", "max_depth", "min_samples_leaf",
        "max_features", "bootstrap", "class_weight", "random_state"
    }
):
    raise ValueError("RF differs from the fixed model contract.")
if protocol["class_handling"]["sample_weight"] is not False or protocol["class_handling"]["resampling"] is not False:
    raise ValueError("Unexpected weighting or resampling.")

train = DatasetBSplitLoader().load("train")
train_ids = train.record_ids.to_numpy(dtype=str)
train_y = train.y.to_numpy(dtype=np.int8)
n_train = len(train_ids)
if n_train != 205310 or len(np.unique(train_ids)) != n_train:
    raise ValueError("Unexpected Train rows or duplicate IDs.")
if not np.array_equal(np.unique(train_y), [0, 1]):
    raise ValueError("Expected both binary classes in Train.")
if len(cv_manifest["folds"]) != 5:
    raise ValueError("Expected five folds.")

fold_inputs = []
validation_counts = np.zeros(n_train, dtype=np.int8)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold_number, ((expected_fit, expected_hold), fold) in enumerate(
    zip(splitter.split(np.zeros(n_train, dtype=np.uint8), train_y), cv_manifest["folds"]),
    start=1,
):
    if fold["fold"] != fold_number or fold["directory"] != f"fold_{fold_number:02d}":
        raise ValueError(f"Fold {fold_number}: manifest order differs.")
    fold_dir = cv_root / fold["directory"]
    with np.load(fold_dir / "rows.npz", allow_pickle=False) as rows:
        required = {
            "train_positions", "fold_validation_positions",
            "train_record_ids", "fold_validation_record_ids",
            "train_y", "fold_validation_y"
        }
        if set(rows.files) != required:
            raise ValueError(f"Fold {fold_number}: row metadata keys differ.")
        fit_positions = rows["train_positions"]
        hold_positions = rows["fold_validation_positions"]
        fit_ids = rows["train_record_ids"]
        hold_ids = rows["fold_validation_record_ids"]
        fit_y = rows["train_y"]
        hold_y = rows["fold_validation_y"]
    if not np.array_equal(fit_positions, expected_fit) or not np.array_equal(hold_positions, expected_hold):
        raise ValueError(f"Fold {fold_number}: positions differ from the locked splitter.")
    if (
        not np.array_equal(train_ids[fit_positions], fit_ids)
        or not np.array_equal(train_ids[hold_positions], hold_ids)
        or not np.array_equal(train_y[fit_positions], fit_y)
        or not np.array_equal(train_y[hold_positions], hold_y)
    ):
        raise ValueError(f"Fold {fold_number}: IDs or labels differ from Train.")
    if len(fit_positions) != fold["train_rows"] or len(hold_positions) != fold["fold_validation_rows"]:
        raise ValueError(f"Fold {fold_number}: row count mismatch.")
    if len(np.unique(fit_y)) != 2 or len(np.unique(hold_y)) != 2:
        raise ValueError(f"Fold {fold_number}: a target class is missing.")
    x_fit_path = fold_dir / "tree_train.npz"
    x_hold_path = fold_dir / "tree_fold_validation.npz"
    x_fit = load_npz(x_fit_path)
    x_hold = load_npz(x_hold_path)
    if x_fit.shape != (len(fit_positions), 66) or x_hold.shape != (len(hold_positions), 66):
        raise ValueError(f"Fold {fold_number}: matrix shape mismatch.")
    if not np.isfinite(x_fit.data).all() or not np.isfinite(x_hold.data).all():
        raise ValueError(f"Fold {fold_number}: non-finite matrix value.")
    validation_counts[hold_positions] += 1
    fold_inputs.append({
        "fold": fold_number, "fit_positions": fit_positions,
        "hold_positions": hold_positions, "fit_y": fit_y,
        "hold_y": hold_y, "x_fit_path": x_fit_path, "x_hold_path": x_hold_path,
    })
    print(f"Preflight fold {fold_number}: {len(fit_y):,} train / {len(hold_y):,} validation, 66 columns")

if not np.all(validation_counts == 1):
    raise ValueError("Every Train record must appear once as fold validation.")
source_hashes = {
    "model_protocol_v1.yaml": hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
    "feature_set_v1.yaml": hashlib.sha256(feature_path.read_bytes()).hexdigest(),
    "split_v1.yaml": hashlib.sha256(split_path.read_bytes()).hexdigest(),
    "cv_preprocessing_manifest.json": hashlib.sha256(cv_manifest_path.read_bytes()).hexdigest(),
}
artifact_dir = PROJECT_ROOT / "artifacts/dataset_b/rf_baseline_v1"
results_path = artifact_dir / "cv_results.json"
oof_path = artifact_dir / "cv_oof_predictions.csv"
print(f"Preflight complete: {n_train:,} Train rows and five valid folds.")
print(f"Fixed RF parameters: {rf_parameters}")


Preflight fold 1: 164,248 train / 41,062 validation, 66 columns
Preflight fold 2: 164,248 train / 41,062 validation, 66 columns
Preflight fold 3: 164,248 train / 41,062 validation, 66 columns
Preflight fold 4: 164,248 train / 41,062 validation, 66 columns
Preflight fold 5: 164,248 train / 41,062 validation, 66 columns
Preflight complete: 205,310 Train rows and five valid folds.
Fixed RF parameters: {'n_estimators': 100, 'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None, 'random_state': 42}


## 2. Fit RF folds and save OOF results

Fit one fresh fixed-parameter Random Forest per fold. Record ROC-AUC, fit/predict time and warnings. Save the complete CV summary and Train OOF probabilities only after all five folds succeed. If matching artifacts already exist, verify and reuse them without fitting.

In [2]:
if results_path.exists() != oof_path.exists():
    raise FileExistsError("Only one RF CV artifact exists; inspect before retrying.")

if results_path.exists():
    with results_path.open(encoding="utf-8") as f:
        cv_results = json.load(f)
    cv_oof_predictions = pd.read_csv(oof_path, dtype={"record_id": str}, float_precision="round_trip")
    expected_folds = np.zeros(n_train, dtype=np.int8)
    for fold in fold_inputs:
        expected_folds[fold["hold_positions"]] = fold["fold"]
    if (
        cv_results.get("model") != "rf"
        or cv_results.get("status") not in ("complete", "complete_with_warnings")
        or cv_results.get("protocol_version") != protocol["protocol_version"]
        or cv_results.get("feature_version") != feature_contract["feature_version"]
        or cv_results.get("split_version") != split_contract["split_version"]
        or cv_results.get("preprocessing_version") != cv_manifest["preprocessing_version"]
        or cv_results.get("parameters") != rf_parameters
        or cv_results.get("source_sha256") != source_hashes
        or cv_results.get("fold_count") != 5
        or cv_results.get("train_rows") != n_train
        or len(cv_results.get("folds", [])) != 5
        or list(cv_oof_predictions.columns) != ["record_id", "y_approved", "fold", "p_approved"]
        or len(cv_oof_predictions) != n_train
        or not np.array_equal(cv_oof_predictions["record_id"].to_numpy(), train_ids)
        or not np.array_equal(cv_oof_predictions["y_approved"].to_numpy(), train_y)
        or not np.array_equal(cv_oof_predictions["fold"].to_numpy(), expected_folds)
    ):
        raise ValueError("Existing RF CV artifacts differ from the Train contract.")
    probabilities = cv_oof_predictions["p_approved"].to_numpy(dtype=float)
    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("Invalid existing OOF probabilities.")
    auc_values = []
    for fold, result in zip(fold_inputs, cv_results["folds"]):
        hold = fold["hold_positions"]
        auc = float(roc_auc_score(train_y[hold], probabilities[hold]))
        if result["fold"] != fold["fold"] or not np.isclose(auc, result["roc_auc"], rtol=0, atol=1e-12):
            raise ValueError(f"Fold {fold['fold']}: stored ROC-AUC differs.")
        auc_values.append(auc)
    if (
        not np.isclose(np.mean(auc_values), cv_results["mean_roc_auc"], rtol=0, atol=1e-12)
        or not np.isclose(np.std(auc_values, ddof=0), cv_results["std_roc_auc"], rtol=0, atol=1e-12)
        or cv_results["std_ddof"] != 0
    ):
        raise ValueError("Stored summary differs from OOF probabilities.")
    print("Existing CV artifacts verified; model fit skipped.")
else:
    oof_probability = np.full(n_train, np.nan, dtype=float)
    oof_fold = np.zeros(n_train, dtype=np.int8)
    fold_results = []
    for fold in fold_inputs:
        number = fold["fold"]
        x_fit = load_npz(fold["x_fit_path"])
        x_hold = load_npz(fold["x_hold_path"])
        model = RandomForestClassifier(**rf_parameters)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            start = time.perf_counter()
            model.fit(x_fit, fold["fit_y"])
            fit_seconds = time.perf_counter() - start
            if not np.array_equal(model.classes_, [0, 1]):
                raise ValueError(f"Fold {number}: unexpected class order.")
            start = time.perf_counter()
            probability = model.predict_proba(x_hold)[:, 1]
            predict_seconds = time.perf_counter() - start
        if (
            len(probability) != len(fold["hold_y"])
            or not np.isfinite(probability).all()
            or np.any((probability < 0) | (probability > 1))
        ):
            raise ValueError(f"Fold {number}: invalid probabilities.")
        hold = fold["hold_positions"]
        if np.any(oof_fold[hold] != 0):
            raise ValueError(f"Fold {number}: duplicate OOF position.")
        oof_probability[hold] = probability
        oof_fold[hold] = number
        auc = float(roc_auc_score(fold["hold_y"], probability))
        warning_messages = [f"{w.category.__name__}: {w.message}" for w in caught]
        fold_results.append({
            "fold": number, "train_rows": len(fold["fit_y"]),
            "fold_validation_rows": len(fold["hold_y"]),
            "roc_auc": auc, "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds, "warnings": warning_messages,
        })
        print(
            f"Fold {number}: ROC-AUC={auc:.6f}, fit={fit_seconds:.2f}s, "
            f"predict={predict_seconds:.2f}s, warnings={len(warning_messages)}"
        )
        for message in warning_messages:
            print(f"  WARNING: {message}")
    if not np.all(oof_fold > 0) or not np.isfinite(oof_probability).all():
        raise ValueError("Missing or invalid OOF probability.")
    cv_oof_predictions = pd.DataFrame({
        "record_id": train_ids, "y_approved": train_y,
        "fold": oof_fold, "p_approved": oof_probability,
    })
    auc_values = np.asarray([r["roc_auc"] for r in fold_results], dtype=float)
    cv_results = {
        "status": "complete_with_warnings" if any(r["warnings"] for r in fold_results) else "complete",
        "model": "rf",
        "protocol_version": protocol["protocol_version"],
        "feature_version": feature_contract["feature_version"],
        "split_version": split_contract["split_version"],
        "preprocessing_version": cv_manifest["preprocessing_version"],
        "population": "train_only", "fold_count": 5,
        "train_rows": n_train,
        "mean_roc_auc": float(auc_values.mean()),
        "std_roc_auc": float(auc_values.std(ddof=0)),
        "std_ddof": 0,
        "parameters": rf_parameters,
        "source_sha256": source_hashes,
        "library_versions": {
            "python": sys.version.split()[0], "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": version("scipy"),
            "scikit-learn": version("scikit-learn"),
        },
        "folds": fold_results,
    }
    artifact_dir.mkdir(parents=True, exist_ok=True)
    cv_oof_predictions.to_csv(oof_path, index=False, float_format="%.17g")
    with results_path.open("w", encoding="utf-8") as f:
        json.dump(cv_results, f, ensure_ascii=False, indent=2)
        f.write("\n")
    print(f"Saved {oof_path}")
    print(f"Saved {results_path}")

print(f"CV status: {cv_results['status']}")
print(f"Mean ROC-AUC: {cv_results['mean_roc_auc']:.6f}")
print(f"Std ROC-AUC (ddof=0): {cv_results['std_roc_auc']:.6f}")
display(pd.DataFrame(cv_results["folds"]).drop(columns=["warnings"]))
display(cv_oof_predictions.head())


Fold 1: ROC-AUC=0.835915, fit=459.41s, predict=1.22s, warnings=0
Fold 2: ROC-AUC=0.837080, fit=408.61s, predict=1.76s, warnings=0
Fold 3: ROC-AUC=0.837344, fit=525.48s, predict=1.87s, warnings=0
Fold 4: ROC-AUC=0.835859, fit=501.01s, predict=1.18s, warnings=0
Fold 5: ROC-AUC=0.840698, fit=406.40s, predict=1.82s, warnings=0
Saved d:\FPTU_Study\FA2026\DAP391m_GR12\artifacts\dataset_b\rf_baseline_v1\cv_oof_predictions.csv
Saved d:\FPTU_Study\FA2026\DAP391m_GR12\artifacts\dataset_b\rf_baseline_v1\cv_results.json
CV status: complete
Mean ROC-AUC: 0.837379
Std ROC-AUC (ddof=0): 0.001764


,fold,train_rows,fold_validation_rows,roc_auc,fit_seconds,predict_seconds
0,1,164248,41062,0.835915,459.407882,1.215385
1,2,164248,41062,0.837080,408.614113,1.762895
2,3,164248,41062,0.837344,525.476864,1.869607
3,4,164248,41062,0.835859,501.005224,1.176474
4,5,164248,41062,0.840698,406.402365,1.822641


,record_id,y_approved,fold,p_approved
0,dataset_b-000343629,1,5,0.95
1,dataset_b-000256425,1,4,0.93
2,dataset_b-000342696,1,2,0.96
3,dataset_b-000235315,0,2,0.61
4,dataset_b-000271985,1,5,0.88


## RF-01 result boundary

A successful run writes five fold ROC-AUC values and one out-of-fold probability for each of 205,310 Train record IDs. CV summary includes fixed parameters, versions, source hashes, timings and warnings. The two artifact files are written only after all folds finish. RF refit, Validation threshold selection and Test evaluation belong to later tasks.


## RF-02 — Fit toàn Train và dự báo Validation

Điều kiện đầu vào là RF-01 đã hoàn thành và hai artifact CV còn khớp với source hash. Bước này tạo một pipeline mới gồm preprocessing profile rf và Random Forest cố định trong model_protocol_v1. Pipeline chỉ fit trên 205.310 dòng Train, sau đó tạo P(Approved) cho 43.995 dòng Validation.

Không dùng fitted transformer của fold CV, không tuning, không chọn threshold và không đọc Test. Kết quả gồm fitted pipeline, Validation predictions và run manifest có hash của model, predictions, nguồn config và preprocessing.


In [3]:
import platform
import joblib
from sklearn.pipeline import Pipeline
from src.dataset_b.preprocessing import make_preprocessing_pipeline

rf01_results_path = PROJECT_ROOT / "artifacts/dataset_b/rf_baseline_v1/cv_results.json"
rf01_oof_path = PROJECT_ROOT / "artifacts/dataset_b/rf_baseline_v1/cv_oof_predictions.csv"
with rf01_results_path.open(encoding="utf-8") as f:
    rf01_results = json.load(f)
if (
    rf01_results.get("status") not in ("complete", "complete_with_warnings")
    or rf01_results.get("model") != "rf"
    or rf01_results.get("fold_count") != 5
    or rf01_results.get("train_rows") != 205310
    or rf01_results.get("parameters") != rf_parameters
    or rf01_results.get("source_sha256") != source_hashes
    or not rf01_oof_path.is_file()
):
    raise ValueError("RF-01 has not passed the locked CV gate.")

loader = DatasetBSplitLoader()
full_train = loader.load("train")
validation = loader.load("validation")
expected_features = feature_contract["model_input_raw_columns"]
for partition in (full_train, validation):
    if list(partition.X.columns) != expected_features:
        raise ValueError(f"{partition.name}: raw features differ from the feature contract.")
if len(full_train.X) != 205310 or len(validation.X) != 43995:
    raise ValueError("Unexpected Train or Validation row count.")
if len(set(full_train.record_ids) & set(validation.record_ids)):
    raise ValueError("Train and Validation record IDs overlap.")
for partition in (full_train, validation):
    if not partition.record_ids.is_unique:
        raise ValueError(f"{partition.name}: duplicate record_id.")
forbidden_inputs = {
    "record_id", "action_taken", "y_approved", *feature_contract["audit_only_columns"]
}
if set(expected_features) & forbidden_inputs:
    raise ValueError("Identifier, target, or audit data entered model inputs.")
if (
    protocol["refit"]["population"] != "full_train"
    or protocol["refit"]["reinitialize_preprocessing"] is not True
    or protocol["refit"]["reinitialize_model"] is not True
    or protocol["validation"]["use"] != "threshold_selection"
    or protocol["validation"]["may_refit_preprocessing_or_model"] is not False
):
    raise ValueError("RF-02 refit or Validation contract differs.")

artifact_dir = PROJECT_ROOT / "artifacts/dataset_b/rf_baseline_v1"
model_path = artifact_dir / "rf_full_train.joblib"
prediction_path = artifact_dir / "validation_predictions.csv"
manifest_path = artifact_dir / "run_manifest.json"
artifact_paths = (model_path, prediction_path, manifest_path)
profiles_path = PROJECT_ROOT / "src/dataset_b/preprocessing/profiles.py"
split_manifest_path = PROJECT_ROOT / "artifacts/dataset_b/split_v1.json"

rf02_source_hashes = {
    "model_protocol_v1.yaml": hashlib.sha256(protocol_path.read_bytes()).hexdigest(),
    "feature_set_v1.yaml": hashlib.sha256(feature_path.read_bytes()).hexdigest(),
    "split_v1.yaml": hashlib.sha256(split_path.read_bytes()).hexdigest(),
    "split_v1.json": hashlib.sha256(split_manifest_path.read_bytes()).hexdigest(),
    "preprocessing_profiles.py": hashlib.sha256(profiles_path.read_bytes()).hexdigest(),
}
rf01_hashes = {
    "cv_results.json": hashlib.sha256(rf01_results_path.read_bytes()).hexdigest(),
    "cv_oof_predictions.csv": hashlib.sha256(rf01_oof_path.read_bytes()).hexdigest(),
}

if any(path.exists() for path in artifact_paths):
    if not all(path.is_file() for path in artifact_paths):
        raise FileExistsError("Partial RF-02 artifacts exist; refusing to overwrite.")
    with manifest_path.open(encoding="utf-8") as f:
        run_manifest = json.load(f)
    expected_manifest_values = {
        "dataset": "dataset_b",
        "run_id": "rf_baseline_v1",
        "stage": "full_train_refit_validation_predictions",
        "model": "rf",
        "model_protocol_version": protocol["protocol_version"],
        "feature_version": feature_contract["feature_version"],
        "split_version": split_contract["split_version"],
        "preprocessing_version": cv_manifest["preprocessing_version"],
        "preprocessing_profile": "rf",
        "fit_partition": "train",
        "fit_rows": 205310,
        "prediction_partition": "validation",
        "prediction_rows": 43995,
        "raw_feature_columns": expected_features,
        "encoded_feature_count": 66,
        "model_parameters": rf_parameters,
        "class_order": [0, 1],
        "probability_column": "p_approved",
        "source_sha256": rf02_source_hashes,
        "upstream_rf01_sha256": rf01_hashes,
    }
    for key, expected in expected_manifest_values.items():
        if run_manifest.get(key) != expected:
            raise ValueError(f"Existing RF-02 manifest differs at {key}.")
    current_versions = {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": version("scipy"),
        "scikit-learn": version("scikit-learn"),
        "joblib": joblib.__version__,
    }
    if run_manifest.get("library_versions") != current_versions:
        raise ValueError("Existing RF-02 artifacts use different library versions.")
    model_hash = hashlib.sha256(model_path.read_bytes()).hexdigest()
    prediction_hash = hashlib.sha256(prediction_path.read_bytes()).hexdigest()
    if (
        run_manifest["artifacts"]["fitted_pipeline"]["sha256"] != model_hash
        or run_manifest["artifacts"]["validation_predictions"]["sha256"] != prediction_hash
    ):
        raise ValueError("Existing RF-02 artifact hash mismatch.")
    print("Existing RF-02 artifacts found; fit skipped and artifacts will be verified.")
else:
    artifact_dir.mkdir(parents=True, exist_ok=True)
    stale_temporary = [
        artifact_dir / "rf_full_train.joblib.tmp",
        artifact_dir / "validation_predictions.csv.tmp",
        artifact_dir / "run_manifest.json.tmp",
    ]
    if any(path.exists() for path in stale_temporary):
        raise FileExistsError("Temporary RF-02 artifact exists from an interrupted write.")

    rf_full_pipeline = Pipeline([
        ("preprocess", make_preprocessing_pipeline("rf")),
        ("estimator", RandomForestClassifier(**rf_parameters)),
    ])
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        started = time.perf_counter()
        rf_full_pipeline.fit(full_train.X, full_train.y)
        fit_seconds = time.perf_counter() - started
    warning_messages = [f"{item.category.__name__}: {item.message}" for item in caught]
    if not np.array_equal(rf_full_pipeline.named_steps["estimator"].classes_, [0, 1]):
        raise ValueError("Unexpected RF class order.")
    encoded_probe = rf_full_pipeline.named_steps["preprocess"].transform(validation.X.iloc[:1])
    if encoded_probe.shape != (1, 66) or not np.isfinite(encoded_probe.data).all():
        raise ValueError("RF preprocessing output is not a finite 66-column matrix.")

    started = time.perf_counter()
    validation_probability = rf_full_pipeline.predict_proba(validation.X)[:, 1]
    validation_predict_seconds = time.perf_counter() - started
    if (
        len(validation_probability) != 43995
        or not np.isfinite(validation_probability).all()
        or np.any((validation_probability < 0) | (validation_probability > 1))
    ):
        raise ValueError("Invalid Validation probabilities.")
    validation_predictions = pd.DataFrame({
        "record_id": validation.record_ids.to_numpy(),
        "y_approved": validation.y.to_numpy(),
        "p_approved": validation_probability,
    })
    if not validation_predictions["record_id"].is_unique:
        raise ValueError("Validation predictions contain duplicate record IDs.")

    temp_model = artifact_dir / "rf_full_train.joblib.tmp"
    temp_prediction = artifact_dir / "validation_predictions.csv.tmp"
    temp_manifest = artifact_dir / "run_manifest.json.tmp"
    joblib.dump(rf_full_pipeline, temp_model)
    validation_predictions.to_csv(temp_prediction, index=False, float_format="%.17g")
    model_hash = hashlib.sha256(temp_model.read_bytes()).hexdigest()
    prediction_hash = hashlib.sha256(temp_prediction.read_bytes()).hexdigest()

    run_manifest = {
        "schema_version": 1,
        "dataset": "dataset_b",
        "run_id": "rf_baseline_v1",
        "stage": "full_train_refit_validation_predictions",
        "model": "rf",
        "model_protocol_version": protocol["protocol_version"],
        "feature_version": feature_contract["feature_version"],
        "split_version": split_contract["split_version"],
        "preprocessing_version": cv_manifest["preprocessing_version"],
        "preprocessing_profile": "rf",
        "fit_partition": "train",
        "fit_rows": len(full_train.X),
        "prediction_partition": "validation",
        "prediction_rows": len(validation_predictions),
        "raw_feature_columns": expected_features,
        "encoded_feature_count": 66,
        "model_parameters": rf_parameters,
        "class_order": [0, 1],
        "probability_column": "p_approved",
        "fit_seconds": fit_seconds,
        "validation_predict_seconds": validation_predict_seconds,
        "fit_warnings": warning_messages,
        "source_sha256": rf02_source_hashes,
        "upstream_rf01_sha256": rf01_hashes,
        "library_versions": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": version("scipy"),
            "scikit-learn": version("scikit-learn"),
            "joblib": joblib.__version__,
        },
        "artifacts": {
            "fitted_pipeline": {
                "filename": model_path.name,
                "sha256": model_hash,
            },
            "validation_predictions": {
                "filename": prediction_path.name,
                "sha256": prediction_hash,
            },
            "run_manifest": {
                "filename": manifest_path.name,
                "self_hash_recorded": False,
            },
        },
    }
    with temp_manifest.open("w", encoding="utf-8") as f:
        json.dump(run_manifest, f, ensure_ascii=False, indent=2)
        f.write("\n")
    temp_model.replace(model_path)
    temp_prediction.replace(prediction_path)
    temp_manifest.replace(manifest_path)
    print(f"Full-Train RF fitted in {fit_seconds:.2f}s; warnings={len(warning_messages)}.")
    print(f"Validation probabilities generated in {validation_predict_seconds:.2f}s.")
    print("Created RF fitted pipeline, Validation predictions, and run manifest.")


Dataset B LTV >1000% retained for 42 rows
Dataset B LTV >1000% retained for 7 rows


Full-Train RF fitted in 725.43s; warnings=0.
Validation probabilities generated in 1.75s.
Created RF fitted pipeline, Validation predictions, and run manifest.


In [4]:
loaded_pipeline = joblib.load(model_path)
saved_predictions = pd.read_csv(
    prediction_path,
    dtype={"record_id": str},
    float_precision="round_trip",
)
with manifest_path.open(encoding="utf-8") as f:
    saved_manifest = json.load(f)

if list(saved_predictions.columns) != ["record_id", "y_approved", "p_approved"]:
    raise ValueError("Unexpected Validation prediction columns.")
if (
    len(saved_predictions) != 43995
    or not saved_predictions["record_id"].is_unique
    or not np.array_equal(saved_predictions["record_id"].to_numpy(), validation.record_ids.to_numpy())
    or not np.array_equal(saved_predictions["y_approved"].to_numpy(), validation.y.to_numpy())
):
    raise ValueError("Saved Validation rows, IDs, or labels differ from the partition.")
saved_probability = saved_predictions["p_approved"].to_numpy(dtype=float)
if not np.isfinite(saved_probability).all() or np.any((saved_probability < 0) | (saved_probability > 1)):
    raise ValueError("Saved Validation probabilities are invalid.")
if not np.array_equal(loaded_pipeline.named_steps["estimator"].classes_, [0, 1]):
    raise ValueError("Reloaded RF class order differs.")
reloaded_probability = loaded_pipeline.predict_proba(validation.X)[:, 1]
if not np.array_equal(reloaded_probability, saved_probability):
    raise ValueError("Reloaded RF does not reproduce the saved Validation probabilities exactly.")
if hashlib.sha256(model_path.read_bytes()).hexdigest() != saved_manifest["artifacts"]["fitted_pipeline"]["sha256"]:
    raise ValueError("Fitted pipeline hash differs after reload.")
if hashlib.sha256(prediction_path.read_bytes()).hexdigest() != saved_manifest["artifacts"]["validation_predictions"]["sha256"]:
    raise ValueError("Validation prediction hash differs after reload.")
if saved_manifest["source_sha256"] != rf02_source_hashes or saved_manifest["upstream_rf01_sha256"] != rf01_hashes:
    raise ValueError("RF-02 provenance hashes differ.")

print(f"RF-02 verification passed for {len(saved_predictions):,} ordered Validation records.")
print(f"Model SHA-256: {saved_manifest['artifacts']['fitted_pipeline']['sha256']}")
print(f"Prediction SHA-256: {saved_manifest['artifacts']['validation_predictions']['sha256']}")
display(saved_predictions.head())


Dataset B LTV >1000% retained for 7 rows


RF-02 verification passed for 43,995 ordered Validation records.
Model SHA-256: 36b6e6d6cc43ef0e783da29867b98db5cd912125fe0862527763a4be8ed976b5
Prediction SHA-256: c086df0928de764a80b493ed969ec03dccecc2b1b2e9c8347ad0d00ecf2c1c4d


,record_id,y_approved,p_approved
0,dataset_b-000299096,1,0.95
1,dataset_b-000172161,0,0.13
2,dataset_b-000065193,0,0.20
3,dataset_b-000291201,0,0.10
4,dataset_b-000137115,1,0.69


## RF-02 result boundary

RF-02 ends after fitting a new RF pipeline on Train and producing ordered Validation probabilities. Validation labels are retained only for later threshold selection. No threshold is selected here, the model is not refit on Validation, and Test remains unopened by this notebook stage.


## RF-03 — Chọn và khóa threshold trên Validation

Dùng đúng 999 threshold từ 0.001 đến 0.999, bước 0.001. Dự báo Approve khi p_approved >= threshold và tối đa hóa Balanced Accuracy trên Validation. Nếu nhiều threshold cùng tối đa, chọn threshold gần 0.5 nhất; nếu vẫn hòa, chọn threshold thấp hơn.

Bước này chỉ đọc Validation predictions đã lưu từ RF-02. Không tải fitted model, không fit, không gọi predict_proba và không đọc Test. Metric tại threshold được chọn là kết quả chọn ngưỡng của run experimental feature v0.2, chưa phải đánh giá RQ1 cuối.


In [5]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

rf03_artifact_dir = PROJECT_ROOT / "artifacts/dataset_b/rf_baseline_v1"
rf03_prediction_path = rf03_artifact_dir / "validation_predictions.csv"
rf03_manifest_path = rf03_artifact_dir / "run_manifest.json"
rf03_validation_path = PROJECT_ROOT / "data/dataset_b/split_v1/validation.csv"
rf03_candidates_path = rf03_artifact_dir / "threshold_candidates.csv"
rf03_selection_path = rf03_artifact_dir / "threshold_selection.json"

def rf03_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def rf03_require(condition, message):
    if not condition:
        raise ValueError(message)

with rf03_manifest_path.open(encoding="utf-8") as handle:
    rf03_manifest = json.load(handle)

rf03_require(rf03_manifest.get("run_id") == "rf_baseline_v1", "Unexpected RF run ID.")
rf03_require(rf03_manifest.get("stage") == "full_train_refit_validation_predictions", "RF-02 stage is incomplete.")
rf03_require(rf03_manifest.get("model") == "rf", "Unexpected model family.")
rf03_require(rf03_manifest.get("fit_partition") == "train", "RF-02 was not fit on Train.")
rf03_require(rf03_manifest.get("fit_rows") == 205310, "Unexpected RF-02 Train row count.")
rf03_require(rf03_manifest.get("prediction_partition") == "validation", "RF-02 prediction partition is not Validation.")
rf03_require(rf03_manifest.get("prediction_rows") == 43995, "Unexpected RF-02 Validation row count.")
rf03_require(rf03_manifest.get("feature_version") == "temporary_training_lock_v0_2", "Unexpected feature version.")
rf03_require(rf03_manifest.get("split_version") == "split_v1", "Unexpected split version.")
rf03_require(rf03_manifest.get("preprocessing_version") == "dataset_b_preprocessing_v2", "Unexpected preprocessing version.")
rf03_require(rf03_manifest.get("model_protocol_version") == "model_protocol_v1", "Unexpected model protocol.")
rf03_require(rf03_manifest.get("probability_column") == "p_approved", "Unexpected probability column.")
rf03_require(rf03_manifest.get("class_order") == [0, 1], "Unexpected class order.")

rf03_prediction_hash = rf03_sha256(rf03_prediction_path)
rf03_expected_prediction_hash = rf03_manifest["artifacts"]["validation_predictions"]["sha256"]
rf03_require(rf03_prediction_hash == rf03_expected_prediction_hash, "Validation prediction hash differs from RF-02.")
rf03_current_manifest_hash = rf03_sha256(rf03_manifest_path)
if "test_inference" in rf03_manifest or "rq1_test_evaluation" in rf03_manifest:
    rf03_manifest_hash = rf03_manifest.get("rf02_manifest_sha256")
    rf03_require(
        rf03_manifest_hash is not None,
        "Updated run manifest is missing the immutable RF-02 manifest hash.",
    )
else:
    rf03_manifest_hash = rf03_current_manifest_hash

rf03_predictions = pd.read_csv(
    rf03_prediction_path,
    dtype={"record_id": str},
    float_precision="round_trip",
)
rf03_validation = pd.read_csv(
    rf03_validation_path,
    usecols=["record_id", "y_approved"],
    dtype={"record_id": str, "y_approved": "int8"},
)
rf03_require(
    list(rf03_predictions.columns) == ["record_id", "y_approved", "p_approved"],
    "Unexpected Validation prediction columns.",
)
rf03_require(
    len(rf03_predictions) == len(rf03_validation) == 43995,
    "Validation row count differs.",
)
rf03_require(rf03_predictions["record_id"].is_unique, "Validation predictions contain duplicate IDs.")
rf03_require(
    np.array_equal(rf03_predictions["record_id"].to_numpy(), rf03_validation["record_id"].to_numpy()),
    "Validation record ID order differs.",
)
rf03_require(
    np.array_equal(rf03_predictions["y_approved"].to_numpy(), rf03_validation["y_approved"].to_numpy()),
    "Validation labels differ.",
)

rf03_y = rf03_predictions["y_approved"].to_numpy(dtype=np.int8)
rf03_p = rf03_predictions["p_approved"].to_numpy(dtype=float)
rf03_require(np.array_equal(np.unique(rf03_y), [0, 1]), "Validation must contain both classes.")
rf03_require(np.isfinite(rf03_p).all(), "Validation probabilities contain non-finite values.")
rf03_require(np.all((0 <= rf03_p) & (rf03_p <= 1)), "Validation probabilities fall outside [0, 1].")

rf03_positive = rf03_y == 1
rf03_negative = rf03_y == 0
rf03_n_approved = int(rf03_positive.sum())
rf03_n_denied = int(rf03_negative.sum())
rf03_require((rf03_n_approved, rf03_n_denied) == (32814, 11181), "Validation class counts differ.")

rf03_grid = np.arange(1, 1000, dtype=np.int64)
rf03_thresholds = rf03_grid / 1000.0
rf03_tp = np.asarray(
    [np.count_nonzero(rf03_p[rf03_positive] >= threshold) for threshold in rf03_thresholds],
    dtype=np.int64,
)
rf03_tn = np.asarray(
    [np.count_nonzero(rf03_p[rf03_negative] < threshold) for threshold in rf03_thresholds],
    dtype=np.int64,
)
rf03_fn = rf03_n_approved - rf03_tp
rf03_fp = rf03_n_denied - rf03_tn

# With fixed positive and negative denominators, this integer score ranks
# Balanced Accuracy exactly and avoids floating-point ambiguity in ties.
rf03_score = rf03_tp * rf03_n_denied + rf03_tn * rf03_n_approved
rf03_max_indices = np.flatnonzero(rf03_score == rf03_score.max())
rf03_selected_index = min(
    rf03_max_indices,
    key=lambda index: (
        abs(int(rf03_grid[index]) - 500),
        int(rf03_grid[index]),
    ),
)
rf03_balanced_accuracy = rf03_score / (2 * rf03_n_approved * rf03_n_denied)

rf03_candidates = pd.DataFrame({
    "threshold": rf03_thresholds,
    "tp_approved": rf03_tp,
    "fn_approved": rf03_fn,
    "tn_denied": rf03_tn,
    "fp_denied": rf03_fp,
    "n_approved": rf03_n_approved,
    "n_denied": rf03_n_denied,
    "recall_approved": rf03_tp / rf03_n_approved,
    "recall_denied": rf03_tn / rf03_n_denied,
    "balanced_accuracy": rf03_balanced_accuracy,
    "is_maximum": rf03_score == rf03_score.max(),
    "is_selected": np.arange(len(rf03_grid)) == rf03_selected_index,
})
rf03_require(len(rf03_candidates) == 999, "Threshold candidate table must have 999 rows.")
rf03_require(int(rf03_candidates["is_selected"].sum()) == 1, "Exactly one threshold must be selected.")
rf03_require(
    np.array_equal(rf03_candidates["tp_approved"] + rf03_candidates["fn_approved"], np.full(999, rf03_n_approved)),
    "Approved confusion counts do not reconcile.",
)
rf03_require(
    np.array_equal(rf03_candidates["tn_denied"] + rf03_candidates["fp_denied"], np.full(999, rf03_n_denied)),
    "Denied confusion counts do not reconcile.",
)

rf03_csv = rf03_candidates.to_csv(index=False, float_format="%.17g")
rf03_candidate_hash = hashlib.sha256(rf03_csv.encode("utf-8")).hexdigest()
rf03_selection = {
    "schema_version": 1,
    "run_id": rf03_manifest["run_id"],
    "stage": "validation_threshold_selection",
    "feature_version": rf03_manifest["feature_version"],
    "split_version": rf03_manifest["split_version"],
    "preprocessing_version": rf03_manifest["preprocessing_version"],
    "model_protocol_version": rf03_manifest["model_protocol_version"],
    "prediction_partition": "validation",
    "prediction_file": rf03_prediction_path.name,
    "prediction_sha256": rf03_prediction_hash,
    "run_manifest_file": rf03_manifest_path.name,
    "run_manifest_sha256": rf03_manifest_hash,
    "candidate_file": rf03_candidates_path.name,
    "candidate_sha256": rf03_candidate_hash,
    "candidate_grid": {
        "start": 0.001,
        "stop": 0.999,
        "step": 0.001,
        "inclusive": True,
        "count": 999,
    },
    "prediction_rule": "p_approved >= threshold -> Approve (y_approved=1)",
    "objective": "maximize_validation_balanced_accuracy",
    "tie_break": [
        "minimum_absolute_distance_to_0.5",
        "lower_threshold",
    ],
    "maximizer_count": int(len(rf03_max_indices)),
    "maximizing_thresholds": [
        int(rf03_grid[index]) / 1000
        for index in rf03_max_indices
    ],
    "selected_threshold": int(rf03_grid[rf03_selected_index]) / 1000,
    "selected_metrics": {
        "balanced_accuracy": float(rf03_balanced_accuracy[rf03_selected_index]),
        "recall_approved": float(rf03_tp[rf03_selected_index] / rf03_n_approved),
        "recall_denied": float(rf03_tn[rf03_selected_index] / rf03_n_denied),
    },
    "validation_counts": {
        "rows": int(len(rf03_predictions)),
        "n_approved": rf03_n_approved,
        "n_denied": rf03_n_denied,
        "tp_approved": int(rf03_tp[rf03_selected_index]),
        "fn_approved": int(rf03_fn[rf03_selected_index]),
        "tn_denied": int(rf03_tn[rf03_selected_index]),
        "fp_denied": int(rf03_fp[rf03_selected_index]),
    },
    "evaluation_status": "experimental_feature_v0_2_validation_selection_only",
}
rf03_json = json.dumps(rf03_selection, ensure_ascii=False, indent=2) + "\n"

if rf03_candidates_path.exists() or rf03_selection_path.exists():
    if not (rf03_candidates_path.is_file() and rf03_selection_path.is_file()):
        raise FileExistsError("Partial RF-03 artifacts exist; refusing to overwrite.")
    if rf03_candidates_path.read_bytes() != rf03_csv.encode("utf-8"):
        raise FileExistsError("Existing RF threshold candidates differ; refusing to overwrite.")
    if rf03_selection_path.read_text(encoding="utf-8") != rf03_json:
        raise FileExistsError("Existing RF threshold selection differs; refusing to overwrite.")
    print("Existing RF-03 artifacts match; no overwrite.")
else:
    rf03_candidates_temp = rf03_artifact_dir / "threshold_candidates.csv.tmp"
    rf03_selection_temp = rf03_artifact_dir / "threshold_selection.json.tmp"
    if rf03_candidates_temp.exists() or rf03_selection_temp.exists():
        raise FileExistsError("Temporary RF-03 artifact exists from an interrupted write.")
    rf03_candidates_temp.write_text(rf03_csv, encoding="utf-8", newline="")
    rf03_selection_temp.write_text(rf03_json, encoding="utf-8", newline="")
    rf03_candidates_temp.replace(rf03_candidates_path)
    rf03_selection_temp.replace(rf03_selection_path)
    print("Created RF-03 threshold artifacts.")

rf03_saved_candidates = pd.read_csv(
    rf03_candidates_path,
    float_precision="round_trip",
)
with rf03_selection_path.open(encoding="utf-8") as handle:
    rf03_saved_selection = json.load(handle)
rf03_require(rf03_sha256(rf03_candidates_path) == rf03_saved_selection["candidate_sha256"], "Candidate hash differs.")
rf03_require(len(rf03_saved_candidates) == 999, "Saved candidate row count differs.")
rf03_require(int(rf03_saved_candidates["is_selected"].sum()) == 1, "Saved candidate selection is not unique.")
rf03_saved_selected = rf03_saved_candidates.loc[rf03_saved_candidates["is_selected"]].iloc[0]
rf03_require(
    float(rf03_saved_selected["threshold"]) == rf03_saved_selection["selected_threshold"],
    "Saved selected threshold differs from candidate table.",
)
rf03_require(
    float(rf03_saved_selected["balanced_accuracy"]) == rf03_saved_selection["selected_metrics"]["balanced_accuracy"],
    "Saved Balanced Accuracy differs from candidate table.",
)

print(
    f"Validation: {len(rf03_predictions):,} rows; "
    f"selected threshold {rf03_selection['selected_threshold']:.3f}; "
    f"Balanced Accuracy {rf03_selection['selected_metrics']['balanced_accuracy']:.6f}; "
    f"{rf03_selection['maximizer_count']} maximizer(s)."
)


Created RF-03 threshold artifacts.
Validation: 43,995 rows; selected threshold 0.662; Balanced Accuracy 0.767228; 1 maximizer(s).


## RF-03 result boundary

RF-03 khóa một threshold riêng cho Random Forest bằng Validation. Hai artifact phải tái tạo được threshold từ toàn bộ 999 ứng viên và giữ liên kết hash tới RF-02. Không fit lại preprocessing hoặc estimator, không dự báo lại Validation và không đọc Test. Chỉ sau khi RF-03 chạy thành công mới được chuẩn bị RF-04.


## RF-04 — Dự báo Test đúng một lần

Dùng fitted Random Forest pipeline của RF-02 và threshold `0.662` đã khóa ở RF-03. Trước khi dự báo, cell kiểm tra feature, split, preprocessing, model và toàn bộ hash liên quan.

Test chỉ được gọi `predict_proba` một lần. Prediction được lưu theo `record_id`; bộ 1.000 stratified bootstrap indices dùng chung ở cấp Dataset B được tái sử dụng. Nếu artifact Test đã tồn tại, cell chỉ kiểm tra và đọc lại artifact, không dự báo lần hai.


In [6]:
from pathlib import Path
import hashlib
import json
import time
import joblib
import numpy as np
import pandas as pd
import yaml
from src.dataset_b.preprocessing import DatasetBSplitLoader

rf04_root = PROJECT_ROOT
rf04_dir = rf04_root / "artifacts" / "dataset_b"
rf04_run_dir = rf04_dir / "rf_baseline_v1"
rf04_model_path = rf04_run_dir / "rf_full_train.joblib"
rf04_manifest_path = rf04_run_dir / "run_manifest.json"
rf04_threshold_path = rf04_run_dir / "threshold_selection.json"
rf04_prediction_path = rf04_run_dir / "test_predictions.csv"
rf04_indices_path = rf04_dir / "rq1_test_bootstrap_indices.npz"
rf04_indices_manifest_path = rf04_dir / "rq1_test_bootstrap_manifest.json"
rf04_split_path = rf04_dir / "split_v1.json"

with rf04_manifest_path.open(encoding="utf-8") as handle:
    rf04_run_manifest = json.load(handle)
with rf04_threshold_path.open(encoding="utf-8") as handle:
    rf04_threshold_selection = json.load(handle)
with (rf04_root / "config/dataset_b/model_protocol_v1.yaml").open(encoding="utf-8") as handle:
    rf04_protocol = yaml.safe_load(handle)
with (rf04_root / "config/dataset_b/feature_set_v1.yaml").open(encoding="utf-8") as handle:
    rf04_feature_contract = yaml.safe_load(handle)

assert rf04_run_manifest["run_id"] == rf04_threshold_selection["run_id"] == "rf_baseline_v1"
assert rf04_run_manifest["feature_version"] == rf04_threshold_selection["feature_version"] == rf04_feature_contract["feature_version"]
assert rf04_run_manifest["split_version"] == rf04_threshold_selection["split_version"] == "split_v1"
assert rf04_run_manifest["preprocessing_version"] == rf04_threshold_selection["preprocessing_version"]
assert rf04_run_manifest["model_protocol_version"] == rf04_threshold_selection["model_protocol_version"] == rf04_protocol["protocol_version"]
assert rf04_run_manifest["model_parameters"] == rf04_protocol["models"]["rf"]["parameters"]
assert rf04_protocol["test"]["may_change_preprocessing_model_or_threshold"] is False
assert rf04_threshold_selection["selected_threshold"] == 0.662
assert hashlib.sha256((rf04_run_dir / "validation_predictions.csv").read_bytes()).hexdigest() == rf04_threshold_selection["prediction_sha256"]
assert hashlib.sha256((rf04_run_dir / "threshold_candidates.csv").read_bytes()).hexdigest() == rf04_threshold_selection["candidate_sha256"]

rf04_test = DatasetBSplitLoader().load("test")
assert len(rf04_test.X) == 43996
assert list(rf04_test.X.columns) == rf04_run_manifest["raw_feature_columns"]
assert len(rf04_test.X.columns) == 12
assert not set(rf04_test.X.columns) & set(rf04_test.audit.columns)
rf04_ids = rf04_test.record_ids.to_numpy()
rf04_y = rf04_test.y.to_numpy(dtype=np.int8)
assert len(set(rf04_ids)) == len(rf04_ids)
assert set(np.unique(rf04_y)) == {0, 1}
rf04_ids_hash = hashlib.sha256(("\n".join(rf04_ids) + "\n").encode("utf-8")).hexdigest()
rf04_model_hash = hashlib.sha256(rf04_model_path.read_bytes()).hexdigest()
rf04_threshold_hash = hashlib.sha256(rf04_threshold_path.read_bytes()).hexdigest()
rf04_current_manifest_hash = hashlib.sha256(rf04_manifest_path.read_bytes()).hexdigest()
if "test_inference" in rf04_run_manifest or "rq1_test_evaluation" in rf04_run_manifest:
    assert rf04_run_manifest["rf02_manifest_sha256"] == rf04_threshold_selection["run_manifest_sha256"]
else:
    assert rf04_current_manifest_hash == rf04_threshold_selection["run_manifest_sha256"]
assert rf04_model_hash == rf04_run_manifest["artifacts"]["fitted_pipeline"]["sha256"]

# Reuse the Dataset B level index matrix already locked by LR-04.
if not (rf04_indices_path.is_file() and rf04_indices_manifest_path.is_file()):
    raise FileNotFoundError("Shared Test bootstrap artifacts are required for RF-04.")
with rf04_indices_manifest_path.open(encoding="utf-8") as handle:
    rf04_indices_manifest = json.load(handle)
assert rf04_indices_manifest["test_record_id_sha256"] == rf04_ids_hash
assert rf04_indices_manifest["split_version"] == "split_v1"
assert rf04_indices_manifest["shape"] == [1000, len(rf04_y)]
assert rf04_indices_manifest["replicates"] == 1000
assert rf04_indices_manifest["seed"] == 42
assert rf04_indices_manifest["class_counts"] == {
    "0": int((rf04_y == 0).sum()),
    "1": int((rf04_y == 1).sum()),
}
assert rf04_indices_manifest["reuse"] == ["lr", "rf", "xgboost"]
assert hashlib.sha256(rf04_indices_path.read_bytes()).hexdigest() == rf04_indices_manifest["indices_sha256"]
print("Reusing shared Dataset B bootstrap indices.")

if rf04_prediction_path.exists():
    if "test_inference" not in rf04_run_manifest:
        raise FileExistsError("Test prediction exists without saved inference metadata; refusing a second inference.")
    rf04_predictions = pd.read_csv(rf04_prediction_path, dtype={"record_id": str}, float_precision="round_trip")
    assert hashlib.sha256(rf04_prediction_path.read_bytes()).hexdigest() == rf04_run_manifest["test_inference"]["prediction_sha256"]
    assert rf04_run_manifest["test_inference"]["model_sha256"] == rf04_model_hash
    assert rf04_run_manifest["test_inference"]["threshold_sha256"] == rf04_threshold_hash
    print("Reusing saved Test predictions; no model inference.")
else:
    if "test_inference" in rf04_run_manifest:
        raise FileExistsError("Inference metadata exists without Test prediction file.")
    rf04_pipeline = joblib.load(rf04_model_path)
    rf04_estimator = rf04_pipeline.named_steps["estimator"]
    assert type(rf04_estimator).__name__ == "RandomForestClassifier"
    assert np.array_equal(rf04_estimator.classes_, [0, 1])
    assert int(rf04_estimator.n_features_in_) == rf04_run_manifest["encoded_feature_count"] == 66
    assert len(rf04_pipeline.named_steps["preprocess"].get_feature_names_out()) == 66
    for rf04_key, rf04_value in rf04_run_manifest["model_parameters"].items():
        assert rf04_estimator.get_params()[rf04_key] == rf04_value
    rf04_start = time.perf_counter()
    rf04_p = rf04_pipeline.predict_proba(rf04_test.X)[:, 1]
    rf04_inference_seconds = time.perf_counter() - rf04_start
    assert len(rf04_p) == len(rf04_y)
    assert np.isfinite(rf04_p).all() and np.all((rf04_p >= 0) & (rf04_p <= 1))
    rf04_predicted = (rf04_p >= 0.662).astype(np.int8)
    rf04_predictions = pd.DataFrame({
        "record_id": rf04_ids,
        "y_approved": rf04_y,
        "p_approved": rf04_p,
        "p_denied": 1.0 - rf04_p,
        "predicted_approved": rf04_predicted,
        "predicted_denied": 1 - rf04_predicted,
    })
    rf04_predictions.to_csv(rf04_prediction_path, index=False, float_format="%.17g")
    rf04_run_manifest["rf02_manifest_sha256"] = rf04_threshold_selection["run_manifest_sha256"]
    rf04_run_manifest["test_inference"] = {
        "partition": "test",
        "rows": len(rf04_y),
        "model_sha256": rf04_model_hash,
        "threshold_sha256": rf04_threshold_hash,
        "selected_threshold": 0.662,
        "prediction_file": rf04_prediction_path.name,
        "prediction_sha256": hashlib.sha256(rf04_prediction_path.read_bytes()).hexdigest(),
        "bootstrap_indices_file": rf04_indices_path.name,
        "bootstrap_indices_sha256": rf04_indices_manifest["indices_sha256"],
        "inference_seconds_including_preprocessing": rf04_inference_seconds,
        "predict_proba_calls_on_test": 1,
    }
    rf04_manifest_path.write_text(json.dumps(rf04_run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8", newline="")
    print(f"Saved one RF Test prediction pass: {len(rf04_y):,} rows in {rf04_inference_seconds:.3f}s.")

assert list(rf04_predictions.columns) == ["record_id", "y_approved", "p_approved", "p_denied", "predicted_approved", "predicted_denied"]
assert np.array_equal(rf04_predictions["record_id"].to_numpy(), rf04_ids)
assert np.array_equal(rf04_predictions["y_approved"].to_numpy(), rf04_y)
assert np.isfinite(rf04_predictions["p_approved"]).all()
assert np.allclose(rf04_predictions["p_denied"], 1.0 - rf04_predictions["p_approved"], rtol=0, atol=1e-15)
assert np.array_equal(rf04_predictions["predicted_approved"], rf04_predictions["p_approved"] >= 0.662)


Reusing shared Dataset B bootstrap indices.


Dataset B LTV >1000% retained for 11 rows


Saved one RF Test prediction pass: 43,996 rows in 2.247s.


## RF-04 — RQ1 point metrics và 95% bootstrap CI

Tính ROC-AUC trên P(Approved), average precision cho Denied, các metric phân lớp tại threshold cố định, Brier, calibration và thời gian. Calibration curve dùng 10 quantile bins; intercept là logistic offset với slope bằng 1, còn slope từ logistic recalibration có intercept tự do. CI percentile 2.5% và 97.5% từ 1.000 bootstrap replicates cho scalar predictive metrics. Invalid metric được ghi theo replicate và loại khỏi CI của chính metric đó. Curve và thời gian không nhận CI từ bootstrap prediction. Kết quả là RQ1 evaluation của rf_baseline_v1 với experimental feature lock v0.2, không phải FINAL_MODEL_FEATURES của toàn dự án.


In [7]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize, root
from scipy.special import expit, logit
from sklearn.metrics import average_precision_score, roc_auc_score

rf04_run_dir = PROJECT_ROOT / "artifacts" / "dataset_b" / "rf_baseline_v1"
rf04_shared_dir = PROJECT_ROOT / "artifacts" / "dataset_b"
rf04_manifest_path = rf04_run_dir / "run_manifest.json"
rf04_prediction_path = rf04_run_dir / "test_predictions.csv"
rf04_metrics_path = rf04_run_dir / "rq1_test_metrics.json"
rf04_bootstrap_path = rf04_run_dir / "rq1_bootstrap_metrics.csv"
rf04_indices_path = rf04_shared_dir / "rq1_test_bootstrap_indices.npz"
rf04_indices_manifest_path = rf04_shared_dir / "rq1_test_bootstrap_manifest.json"
with rf04_manifest_path.open(encoding="utf-8") as handle:
    rf04_run_manifest = json.load(handle)
with rf04_indices_manifest_path.open(encoding="utf-8") as handle:
    rf04_indices_manifest = json.load(handle)
rf04_prediction_sha = hashlib.sha256(rf04_prediction_path.read_bytes()).hexdigest()
assert rf04_prediction_sha == rf04_run_manifest["test_inference"]["prediction_sha256"]
assert hashlib.sha256(rf04_indices_path.read_bytes()).hexdigest() == rf04_indices_manifest["indices_sha256"]
assert rf04_run_manifest["test_inference"]["bootstrap_indices_sha256"] == rf04_indices_manifest["indices_sha256"]
assert rf04_run_manifest["test_inference"]["predict_proba_calls_on_test"] == 1
rf04_predictions = pd.read_csv(rf04_prediction_path, dtype={"record_id": str}, float_precision="round_trip")
assert list(rf04_predictions.columns) == ["record_id", "y_approved", "p_approved", "p_denied", "predicted_approved", "predicted_denied"]
assert len(rf04_predictions) == rf04_run_manifest["test_inference"]["rows"] == rf04_indices_manifest["test_rows"] == 43996
assert rf04_predictions["record_id"].is_unique
rf04_y = rf04_predictions["y_approved"].to_numpy(dtype=np.int8)
rf04_p = rf04_predictions["p_approved"].to_numpy(dtype=np.float64)
rf04_threshold = float(rf04_run_manifest["test_inference"]["selected_threshold"])
assert rf04_threshold == 0.662
assert set(np.unique(rf04_y)) == {0, 1}
assert np.isfinite(rf04_p).all() and np.all((rf04_p >= 0) & (rf04_p <= 1))
assert np.allclose(rf04_predictions["p_denied"], 1 - rf04_p, rtol=0, atol=1e-15)
assert np.array_equal(rf04_predictions["predicted_approved"], (rf04_p >= rf04_threshold).astype(np.int8))
assert np.array_equal(rf04_predictions["predicted_denied"], (rf04_p < rf04_threshold).astype(np.int8))
rf04_ids_sha = hashlib.sha256(("\n".join(rf04_predictions["record_id"]) + "\n").encode("utf-8")).hexdigest()
assert rf04_ids_sha == rf04_indices_manifest["test_record_id_sha256"]

rf04_scalar_names = [
    "roc_auc_approved", "average_precision_denied",
    "precision_denied", "recall_denied", "f1_denied",
    "recall_approved", "balanced_accuracy", "brier_approved",
    "calibration_intercept", "calibration_slope",
]

def rf04_calibration(y, p):
    z = logit(np.clip(p, 1e-15, 1 - 1e-15))
    target = y.astype(np.float64)
    invalid = {}
    intercept = np.nan
    slope = np.nan
    try:
        intercept = float(brentq(
            lambda a: float(np.mean(expit(z + a) - target)),
            -80, 80, xtol=1e-10,
        ))
    except (ArithmeticError, ValueError, RuntimeError) as exc:
        invalid["calibration_intercept"] = str(exc)

    def loss_and_gradient(theta):
        eta = theta[0] + theta[1] * z
        error = expit(eta) - target
        loss = float(np.mean(np.logaddexp(0, eta) - target * eta))
        gradient = np.array([np.mean(error), np.mean(error * z)], dtype=np.float64)
        return loss, gradient

    try:
        slope_fit = minimize(
            loss_and_gradient,
            x0=np.array([0.0, 1.0]),
            method="L-BFGS-B",
            jac=True,
            options={"maxiter": 100, "ftol": 1e-12, "gtol": 1e-9},
        )
        slope_parameters = slope_fit.x
        if not slope_fit.success or not np.isfinite(slope_parameters).all():
            initial = slope_parameters if np.isfinite(slope_parameters).all() else np.array([0.0, 1.0])
            fallback = root(lambda theta: loss_and_gradient(theta)[1], initial, method="hybr", tol=1e-10)
            slope_parameters = fallback.x
        score = loss_and_gradient(slope_parameters)[1]
        if not np.isfinite(slope_parameters).all() or not np.isfinite(score).all() or np.max(np.abs(score)) > 1e-7:
            raise ArithmeticError("Calibration slope score did not converge.")
        slope = float(slope_parameters[1])
    except (ArithmeticError, ValueError, RuntimeError) as exc:
        invalid["calibration_slope"] = str(exc)
    return intercept, slope, invalid

def rf04_metrics(y, p, threshold):
    invalid = {}
    values = {}
    predicted_approved = p >= threshold
    approved = y == 1
    denied = y == 0
    tp = int(np.count_nonzero(approved & predicted_approved))
    fn = int(np.count_nonzero(approved & ~predicted_approved))
    tn = int(np.count_nonzero(denied & ~predicted_approved))
    fp = int(np.count_nonzero(denied & predicted_approved))
    n_approved = tp + fn
    n_denied = tn + fp
    if n_approved == 0 or n_denied == 0:
        raise ArithmeticError("Both true classes are required.")
    values["roc_auc_approved"] = float(roc_auc_score(y, p))
    values["average_precision_denied"] = float(average_precision_score(1 - y, 1 - p))
    if tn + fn:
        values["precision_denied"] = tn / (tn + fn)
    else:
        values["precision_denied"] = np.nan
        invalid["precision_denied"] = "no_predicted_denied"
    values["recall_denied"] = tn / n_denied
    values["f1_denied"] = 2 * tn / (2 * tn + fn + fp)
    values["recall_approved"] = tp / n_approved
    values["balanced_accuracy"] = (values["recall_approved"] + values["recall_denied"]) / 2
    values["brier_approved"] = float(np.mean((p - y) ** 2))
    values["calibration_intercept"], values["calibration_slope"], calibration_invalid = rf04_calibration(y, p)
    invalid.update(calibration_invalid)
    for name, value in values.items():
        if not np.isfinite(value):
            invalid.setdefault(name, "nonfinite_metric")
    return values, invalid, {
        "tp_approved": tp, "fn_approved": fn,
        "tn_denied": tn, "fp_denied": fp,
        "n_approved": n_approved, "n_denied": n_denied,
    }

def rf04_curve(y, p):
    edges = np.unique(np.quantile(p, np.linspace(0, 1, 11), method="linear"))
    if len(edges) < 2:
        raise ArithmeticError("Calibration curve has fewer than two distinct edges.")
    bin_id = np.searchsorted(edges[1:-1], p, side="right")
    bins = []
    for i in range(len(edges) - 1):
        selected = bin_id == i
        if not selected.any():
            continue
        bins.append({
            "bin": i,
            "lower_edge": float(edges[i]),
            "upper_edge": float(edges[i + 1]),
            "count": int(selected.sum()),
            "mean_p_approved": float(p[selected].mean()),
            "observed_approval_rate": float(y[selected].mean()),
        })
    assert sum(item["count"] for item in bins) == len(y)
    return bins

if rf04_metrics_path.exists() or rf04_bootstrap_path.exists():
    if not (rf04_metrics_path.is_file() and rf04_bootstrap_path.is_file()):
        raise FileExistsError("Partial RF-04 metric artifacts exist.")
    evaluation = rf04_run_manifest.get("rq1_test_evaluation")
    if evaluation is None:
        raise FileExistsError("Metric artifacts exist without run manifest references.")
    assert hashlib.sha256(rf04_metrics_path.read_bytes()).hexdigest() == evaluation["metrics_sha256"]
    assert hashlib.sha256(rf04_bootstrap_path.read_bytes()).hexdigest() == evaluation["bootstrap_metrics_sha256"]
    print("Existing RF-04 metrics match; no recomputation.")
else:
    point, point_invalid, counts = rf04_metrics(rf04_y, rf04_p, rf04_threshold)
    if point_invalid:
        raise ArithmeticError(f"Test point metrics are invalid: {point_invalid}")
    curve = rf04_curve(rf04_y, rf04_p)
    with np.load(rf04_indices_path, allow_pickle=False) as package:
        indices = package["indices"]
    assert indices.shape == (1000, len(rf04_y))
    assert indices.dtype == np.int32
    assert np.all((indices >= 0) & (indices < len(rf04_y)))
    n_denied = counts["n_denied"]
    assert np.all(rf04_y[indices[:, :n_denied]] == 0)
    assert np.all(rf04_y[indices[:, n_denied:]] == 1)
    rows = []
    for replicate_id, row_indices in enumerate(indices):
        sampled, invalid, _ = rf04_metrics(rf04_y[row_indices], rf04_p[row_indices], rf04_threshold)
        rows.append({
            "replicate_id": replicate_id,
            **sampled,
            "invalid_metrics": json.dumps(invalid, ensure_ascii=False, sort_keys=True),
        })
        if (replicate_id + 1) % 100 == 0:
            print(f"Bootstrap {replicate_id + 1}/1000", flush=True)
    bootstrap_frame = pd.DataFrame(rows, columns=["replicate_id", *rf04_scalar_names, "invalid_metrics"])
    assert len(bootstrap_frame) == 1000
    ci = {}
    for name in rf04_scalar_names:
        values = bootstrap_frame[name].to_numpy(dtype=np.float64)
        valid = values[np.isfinite(values)]
        ci[name] = {
            "lower_2_5": float(np.quantile(valid, 0.025, method="linear")) if len(valid) else None,
            "upper_97_5": float(np.quantile(valid, 0.975, method="linear")) if len(valid) else None,
            "valid_replicates": int(len(valid)),
            "invalid_replicates": int(len(values) - len(valid)),
        }
    bootstrap_csv = bootstrap_frame.to_csv(index=False, float_format="%.17g")
    metrics_payload = {
        "schema_version": 1,
        "run_id": rf04_run_manifest["run_id"],
        "evaluation_status": "complete_rq1_evaluation_experimental_feature_v0_2" if all(item["valid_replicates"] >= 2 for item in ci.values()) else "partial_ci",
        "feature_version": rf04_run_manifest["feature_version"],
        "split_version": rf04_run_manifest["split_version"],
        "preprocessing_version": rf04_run_manifest["preprocessing_version"],
        "model_protocol_version": rf04_run_manifest["model_protocol_version"],
        "partition": "test",
        "rows": len(rf04_y),
        "threshold": rf04_threshold,
        "prediction_file": rf04_prediction_path.name,
        "prediction_sha256": rf04_prediction_sha,
        "model_sha256": rf04_run_manifest["test_inference"]["model_sha256"],
        "bootstrap_indices_file": rf04_indices_path.name,
        "bootstrap_indices_sha256": rf04_indices_manifest["indices_sha256"],
        "bootstrap_replicates_file": rf04_bootstrap_path.name,
        "bootstrap_replicates_sha256": hashlib.sha256(bootstrap_csv.encode("utf-8")).hexdigest(),
        "counts": counts,
        "point_metrics": point,
        "confidence_intervals_95_percentile": ci,
        "calibration_curve": curve,
        "computational_metrics_seconds": {
            "full_train_fit": rf04_run_manifest["fit_seconds"],
            "test_inference_including_preprocessing": rf04_run_manifest["test_inference"]["inference_seconds_including_preprocessing"],
        },
        "methods": {
            "roc_auc_approved": "sklearn roc_auc_score(y_approved, p_approved)",
            "average_precision_denied": "sklearn average_precision_score(1-y_approved, 1-p_approved)",
            "predicted_approved": "p_approved >= locked threshold",
            "brier_approved": "mean((p_approved-y_approved)^2)",
            "calibration_curve": "10 quantile bins from Test p_approved; duplicate edges collapsed; no bootstrap CI",
            "calibration_intercept": "logistic offset recalibration of y_approved on logit(clip(p,1e-15,1-1e-15)); slope fixed at 1",
            "calibration_slope": "unpenalized logistic recalibration of y_approved on logit(clip(p,1e-15,1-1e-15)); free intercept and slope",
            "bootstrap": "1000 shared stratified true-class row-index replicates, seed 42, model and threshold fixed; 2.5 and 97.5 empirical percentiles",
            "invalid_replicate": "record reason in invalid_metrics; exclude only the undefined scalar metric from its CI; do not resample",
            "timing_ci": "no prediction bootstrap CI for fit or inference time",
        },
    }
    metrics_json = json.dumps(metrics_payload, ensure_ascii=False, indent=2) + "\n"
    rf04_bootstrap_path.write_text(bootstrap_csv, encoding="utf-8", newline="")
    rf04_metrics_path.write_text(metrics_json, encoding="utf-8", newline="")
    rf04_run_manifest["rq1_test_evaluation"] = {
        "status": metrics_payload["evaluation_status"],
        "metrics_file": rf04_metrics_path.name,
        "metrics_sha256": hashlib.sha256(rf04_metrics_path.read_bytes()).hexdigest(),
        "bootstrap_metrics_file": rf04_bootstrap_path.name,
        "bootstrap_metrics_sha256": hashlib.sha256(rf04_bootstrap_path.read_bytes()).hexdigest(),
        "bootstrap_indices_file": rf04_indices_path.name,
        "bootstrap_indices_sha256": rf04_indices_manifest["indices_sha256"],
    }
    rf04_manifest_path.write_text(json.dumps(rf04_run_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8", newline="")
    invalid_total = sum(item["invalid_replicates"] for item in ci.values())
    print(f"Saved RF-04 metrics and 95% CIs. Invalid metric-replicate values: {invalid_total}.")


Bootstrap 100/1000
Bootstrap 200/1000
Bootstrap 300/1000
Bootstrap 400/1000
Bootstrap 500/1000
Bootstrap 600/1000
Bootstrap 700/1000
Bootstrap 800/1000
Bootstrap 900/1000
Bootstrap 1000/1000
Saved RF-04 metrics and 95% CIs. Invalid metric-replicate values: 1.


## RF-04 — Quy ước kỹ thuật đã dùng

### Phạm vi run

| Thành phần | Giá trị |
| --- | --- |
| Model | `rf_baseline_v1` |
| Feature | Experimental feature lock v0.2 |
| Split | `split_v1` |
| Preprocessing | `dataset_b_preprocessing_v2`, profile RF |
| Đầu vào | 12 raw features |
| Sau mã hóa | 66 features |
| Phạm vi kết luận | Complete RQ1 evaluation của RF v1; chưa phải `FINAL_MODEL_FEATURES` |

### Ranh giới dữ liệu và dự báo

| Partition | Cách sử dụng |
| --- | --- |
| Train | Fit pipeline và Random Forest |
| Validation | Chọn threshold theo Balanced Accuracy |
| Test | Dự báo đúng một lần bằng pipeline đã khóa; không refit hoặc retune |

- `y_approved = 1`: Approved; `y_approved = 0`: Denied.
- `p_approved = P(Approved)` và `p_denied = 1 - p_approved`.
- Quy tắc phân lớp: `p_approved >= 0.662` → Approved; còn lại → Denied.
- RF-03 duyệt threshold từ `0.001` đến `0.999`, bước `0.001`; khi hòa chọn threshold gần `0.5` nhất, sau đó chọn threshold thấp hơn.

### Định nghĩa metric

| Nhóm | Cách tính |
| --- | --- |
| Discrimination | ROC-AUC trên `y_approved` và `p_approved` |
| Denied ranking | Average precision trên `1-y_approved` và `1-p_approved` |
| Denied classification | Precision, Recall và F1 tại threshold đã khóa |
| Approved classification | Recall Approved tại threshold đã khóa |
| Balanced Accuracy | Trung bình Recall Approved và Recall Denied |
| Probability quality | Brier = trung bình `(p_approved - y_approved)^2` |

### Calibration và độ bất định

| Thành phần | Quy ước |
| --- | --- |
| Calibration curve | 10 quantile bins; cạnh trùng được gộp |
| Xử lý xác suất | Clip vào `[1e-15, 1-1e-15]` trước khi lấy logit |
| Calibration intercept | Logistic offset, slope cố định bằng 1 |
| Calibration slope | Logistic recalibration không regularization, intercept tự do |
| Bootstrap | 1.000 mẫu phân tầng theo lớp thật, lấy mẫu có hoàn lại, seed 42 |
| CI | Percentile 95% `[2,5%; 97,5%]`, NumPy quantile `method="linear"` |
| Invalid metric | Ghi lý do và loại riêng metric đó; không lấy mẫu bù |
| Không bootstrap | Calibration curve, training time và inference time |

Bootstrap indices được dùng chung cho LR, RF và XGBoost để hỗ trợ paired comparison sau này. Model và threshold được giữ cố định trong mọi replicate.

### Thời gian và truy vết

- Full-Train fit time lấy từ `run_manifest.json`.
- Test inference time bao gồm preprocessing và một lần `predict_proba`; không gồm đọc/ghi file.
- Artifact nguồn: `../../artifacts/dataset_b/rf_baseline_v1/run_manifest.json`.
- Metric đầy đủ: `../../artifacts/dataset_b/rf_baseline_v1/rq1_test_metrics.json`.
- Bootstrap dùng chung: `../../artifacts/dataset_b/rq1_test_bootstrap_manifest.json`.


## RF v1 — Các quyết định mặc định và vấn đề cần ghi nhớ

Các lựa chọn dưới đây đã được khóa để RF dùng cùng hợp đồng đánh giá với LR. Chúng không được thay đổi dựa trên kết quả Test.

| Hạng mục | Quyết định đã áp dụng | Ý nghĩa |
| --- | --- | --- |
| Cấu hình model | Dùng nguyên RF baseline trong `model_protocol_v1.yaml` | Không tuning và không tìm cấu hình tốt nhất trên Validation/Test |
| Mất cân bằng lớp | Không class weighting, resampling hoặc sample weighting | Giữ baseline cố định để so sánh |
| Chọn threshold | Tối đa Balanced Accuracy trên Validation | Test không tham gia chọn threshold |
| Xử lý đồng hạng | Gần `0.5` nhất; nếu vẫn hòa chọn threshold thấp hơn | Cho kết quả xác định và tái lập được |
| Nhãn dự báo | `p_approved >= threshold` → Approved | Đồng nhất với LR và các model tiếp theo |
| Bootstrap | 1.000 stratified true-class replicates, seed 42 | Giữ nguyên cỡ hai lớp trong từng replicate |
| CI | 95% percentile, NumPy quantile `linear` | Đồng nhất cách báo CI giữa các model |
| Replicate không hợp lệ | Loại riêng metric lỗi, ghi lý do, không resample | Không che giấu lỗi hội tụ |
| Calibration bins | Quantile bins; gộp cạnh trùng | RF có xác suất rời rạc nên số bin thực tế có thể dưới 10 |
| Phạm vi kết luận | Experimental feature lock v0.2 | Chưa phải kết luận model hoặc feature cuối cùng |

### Vấn đề quan sát được

- Calibration slope trên Test là **0.286269**, CI 95% **[0.248607; 0.323426]**. Xác suất RF có xu hướng quá cực đoan so với tỷ lệ quan sát; cần giữ điểm này khi so sánh calibration giữa các model.
- Calibration curve có **9 bin thực tế** vì một số cạnh quantile trùng nhau. Đây là kết quả đúng theo quy tắc gộp cạnh, không phải thiếu dữ liệu.
- Có **1/1.000** bootstrap replicate không hội tụ cho calibration slope. CI của slope dùng 999 replicate hợp lệ; chín scalar metrics còn lại dùng đủ 1.000.
- Test đã được mở và dự báo đúng một lần cho run này. Mọi thay đổi model, feature hoặc threshold về sau phải tạo run/version mới, không ghi đè kết quả RF v1.


## RF v1 — Bảng kết quả để đối chiếu

Bản chụp tĩnh từ artifact của `rf_baseline_v1`. Nếu artifact được cập nhật, cần tạo lại bảng này. Đây là kết quả của experimental feature lock v0.2, chưa phải kết luận so sánh cuối cùng với LR/XGBoost hoặc `FINAL_MODEL_FEATURES`.

### Train CV và chọn ngưỡng

| Giai đoạn | Chỉ số | Giá trị |
| --- | --- | ---: |
| Train, 5-fold CV | Mean ROC-AUC | 0.837379 |
| Train, 5-fold CV | Std ROC-AUC (ddof=0) | 0.001764 |
| Validation | Threshold đã khóa | 0.662 |
| Validation | Balanced Accuracy tại ngưỡng đã chọn | 0.767228 |
| Validation | Recall Approved | 0.874139 |
| Validation | Recall Denied | 0.660317 |

### Test: point metrics và 95% CI

Test có 43.996 hồ sơ. CI dùng 1.000 shared stratified bootstrap replicates. Calibration slope có 999 giá trị hợp lệ và 1 giá trị bị loại do không hội tụ; các metric còn lại có đủ 1.000 giá trị hợp lệ.

| Nhóm | Metric | Point | CI 95% [2,5%; 97,5%] |
| --- | --- | ---: | ---: |
| Discrimination | ROC-AUC Approved | 0.836716 | [0.831918; 0.841286] |
| Denied | Average precision Denied | 0.742903 | [0.736193; 0.749617] |
| Denied | Precision Denied | 0.640517 | [0.633477; 0.647246] |
| Denied | Recall Denied | 0.655755 | [0.647080; 0.664610] |
| Denied | F1 Denied | 0.648047 | [0.641523; 0.654996] |
| Classification | Recall Approved | 0.874600 | [0.871156; 0.877952] |
| Classification | Balanced Accuracy | 0.765178 | [0.760398; 0.770216] |
| Probability quality | Brier score | 0.117892 | [0.116057; 0.119849] |
| Calibration | Calibration intercept | -0.016461 | [-0.038768; 0.005062] |
| Calibration | Calibration slope | 0.286270 | [0.248607; 0.323426] |

### Ma trận phân lớp và thời gian

| Nhãn thật | Dự báo Approved | Dự báo Denied | Tổng |
| --- | ---: | ---: | ---: |
| Approved | 28.700 | 4.115 | 32.815 |
| Denied | 3.849 | 7.332 | 11.181 |

- Full-Train fit: **725,427 giây**.
- Test inference, bao gồm preprocessing: **2,247 giây**.
- Không tính CI bootstrap cho thời gian.

### Calibration curve trên Test

Do xác suất RF có nhiều giá trị trùng nhau, 10 quantile bins tạo thành 9 bin thực tế sau khi gộp cạnh trùng.

| Bin | Khoảng `p_approved` | Số hồ sơ | P(Approved) trung bình | Tỷ lệ Approved thực tế |
| ---: | --- | ---: | ---: | ---: |
| 1 | [0.000000; 0.150000] | 4.394 | 0.040719 | 0.053027 |
| 2 | [0.150000; 0.540000] | 4.270 | 0.359705 | 0.448946 |
| 3 | [0.540000; 0.720000] | 4.392 | 0.637190 | 0.722905 |
| 4 | [0.720000; 0.819667] | 4.542 | 0.768862 | 0.801189 |
| 5 | [0.819667; 0.880000] | 3.991 | 0.846958 | 0.850664 |
| 6 | [0.880000; 0.929746] | 4.808 | 0.901709 | 0.875832 |
| 7 | [0.929746; 0.960000] | 4.046 | 0.940920 | 0.894464 |
| 8 | [0.960000; 0.980000] | 3.567 | 0.965699 | 0.915896 |
| 9 | [0.980000; 1.000000] | 9.986 | 0.992632 | 0.937212 |

Nguồn đầy đủ: `../../artifacts/dataset_b/rf_baseline_v1/rq1_test_metrics.json`; replicate-level values: `../../artifacts/dataset_b/rf_baseline_v1/rq1_bootstrap_metrics.csv`. Giá trị hiển thị được làm tròn sáu chữ số thập phân; artifact giữ độ chính xác gốc.
